Check **Setup**

In [34]:
# platform - built-in python library
# gives info about the system we are running on
import platform
import subprocess
import os

In [35]:
print("Python version", platform.python_version())

Python version 3.12.13


In [36]:
# nvidia-smi is a terminal command that checks if an NVIDIA GPU exists
# subprocess.run() runs that command and captures the output

results = subprocess.run(["nvidia-smi"], capture_output=True, text=True)

if results.returncode == 0:
  print("GPU available")
else:
  print("No GPU - using CPU")

GPU available


Mount **Google Drive**

In [5]:
# drive lets us connect our Google Drive to this collab
from google.colab import drive
import os

# os helps to interact with the file system


drive.mount("/content/drive", force_remount=True)

PROJECT_DIR = "/content/drive/MyDrive/neurosynth"

os.makedirs(PROJECT_DIR, exist_ok=True)

# create a raw data folder - download EEG files go here
os.makedirs(f"{PROJECT_DIR}/data/raw", exist_ok=True)

# create the processed data folder - cleaned numpy arrays go here
os.makedirs(f"{PROJECT_DIR}/data/processed", exist_ok=True)

Mounted at /content/drive


Installing packages

In [6]:
# mne is a EEG processing library
# mlflow tracks our training experiments (loss, accuracy, parameters)

!pip install mne torch transformers mlflow scikit-learn -q

In [7]:
import mne  # EEG processing
import torch  # deep learning
import transformers # transformer model building blocks
import sklearn  # ml utilities
import numpy as np

 Connect to GitHub

In [37]:
import subprocess
import os
from google.colab import userdata

GITHUB_TOKEN    = userdata.get("GITHUB_TOKEN")
GITHUB_USERNAME = "the-liyanage"
REPO_NAME       = "neurosynth"

# set git identity — must do this every session
subprocess.run(["git", "config", "--global",
                "user.name", "the-liyanage"])
subprocess.run(["git", "config", "--global",
                "user.email", "hiruniliyanage4@gmail.com"])

# remove the empty folder that's there now
subprocess.run(["rm", "-rf", f"/content/{REPO_NAME}"])
print("Old folder removed")


# clone properly from GitHub — this creates the .git folder
os.chdir("/content")
result = subprocess.run([
    "git", "clone",
    f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
], capture_output=True, text=True)

Repo folder exists in Colab
Old folder removed


Download EEG Data

In [38]:
from mne.datasets import eegbci

# where to save the raw downloaded files (on Google Drive)
DATA_DIR = "/content/drive/MyDrive/neurosynth/data/raw"


# People whose brain signals were recoreded (starts with 5)
SUBJECTS = [1, 2, 3, 4, 5]

# Recording sessions (6, 10, 14 are the motor IMAGERY sessions)
RUNS = [6, 10, 14]

print("Downloading EEG Data.... \n")

for subject in SUBJECTS:
  raw_fnames = eegbci.load_data(
      subject,  # which person
      runs = RUNS, # which session
      path = DATA_DIR,   # where to save the data
      verbose = False  # don't print MNE's internal logs

  )

  # raw_fnames is the list of file paths that were downloaded
  print(f"Subject {subject:03d} - {len(raw_fnames)} files downloaded")


Subject 001 - 3 files downloaded
Subject 002 - 3 files downloaded
Subject 003 - 3 files downloaded
Subject 004 - 3 files downloaded
Subject 005 - 3 files downloaded


In [39]:
# concatenate_raws joins multiple recordings into one
from mne.io import concatenate_raws

# load just subject 1, run 6
raw_fnames = eegbci.load_data(
    1,  # subject number
    runs = [6],
    path = DATA_DIR,
    verbose = False
)


# read_raw_edf reads the .edf file (European Data Format)
# .edf is the standard file format for biological signal recordings
raw = mne.io.read_raw_edf(raw_fnames[0], preload = True, verbose = False)


# ch_names is a list of all electrode names
print(f"Channels: {len(raw.ch_names)}")

# sfreq = sampling frequency = how many readings per second
print(f"Sampe rate:  {raw.info["sfreq"]} Hz")

# raw.times is an array of every
print(f"Duration: {raw.times[-1]:.1f} seconds")

# get_data() returns the raw signal as a numpy array
print(f"Data shape: {raw.get_data().shape}")
print(f"            (channels, timepoints)")


print(f"\nFirst 5 channels: {raw.ch_names[:5]}")


# annotations are the labels - timestamps marking when each task happened
# T0 = rest, T1 = left first imagery, T2, = right first imegery
print(f"\nAnnotations")
for ann in raw.annotations:
  print(f"  {ann['onset']:.1f}s --->'{ann['description']}'")



Channels: 64
Sampe rate:  160.0 Hz
Duration: 125.0 seconds
Data shape: (64, 20000)
            (channels, timepoints)

First 5 channels: ['Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.']

Annotations
  0.0s --->'T0'
  4.2s --->'T2'
  8.3s --->'T0'
  12.5s --->'T1'
  16.6s --->'T0'
  20.8s --->'T1'
  24.9s --->'T0'
  29.1s --->'T2'
  33.2s --->'T0'
  37.4s --->'T1'
  41.5s --->'T0'
  45.7s --->'T2'
  49.8s --->'T0'
  54.0s --->'T2'
  58.1s --->'T0'
  62.3s --->'T1'
  66.4s --->'T0'
  70.6s --->'T1'
  74.7s --->'T0'
  78.9s --->'T2'
  83.0s --->'T0'
  87.2s --->'T2'
  91.3s --->'T0'
  95.5s --->'T1'
  99.6s --->'T0'
  103.8s --->'T1'
  107.9s --->'T0'
  112.1s --->'T2'
  116.2s --->'T0'
  120.4s --->'T2'


In [41]:
import shutil
from datetime import datetime


REPO_NAME = "neurosynth"

# copy the notebook from Google Drive into our cloned repo folder
# source --> our notebook saved on Google Drive
# destination --> notebooks folder inside the cloned repo

# Ensure the destination directory exists
os.makedirs(f"/content/{REPO_NAME}/notebooks", exist_ok=True)

shutil.copy(
    "/content/drive/MyDrive/Colab Notebooks/01_download_data.ipynb",
    f"/content/{REPO_NAME}/notebooks/01_download_data.ipynb"
)

# move into the repo folder
os.chdir(f"/content/{REPO_NAME}")

subprocess.run(["git", "add", "."])

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
subprocess.run(["git", "commit", "-m",
                "add EEG download notebook"])

result = subprocess.run([
    "git", "push",
    f"https://{userdata.get('GITHUB_TOKEN')}@github.com/the-liyanage/neurosynth.git",
     "main"],
    capture_output=True,
    text=True
    )



print(" Pushed!" if result.returncode == 0 else result.stderr)

fatal: not a git repository (or any of the parent directories): .git



In [42]:
import os
print("Current folder:", os.getcwd())
print("Contents of /content:")
print(os.listdir("/content"))

Current folder: /content/neurosynth
Contents of /content:
['.config', 'drive', 'neurosynth', 'sample_data']


In [43]:
contents = os.listdir("/content/neurosynth")
print("Contents of /content/neurosynth:")
print(contents)

Contents of /content/neurosynth:
['notebooks']
